<a href="https://colab.research.google.com/github/areyesp-77/intro_nlp/blob/main/session13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistematización

1.   Análisis de datos
2.   Extracción de características
3.   Clasificación textual



In [ ]:
# Importar librerías
import numpy as np
import sklearn
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files


## Task 1: análisis de datos

- Notas de suicidio.

Identificar los rasgos lingüísticos más discriminantes entre datos positivos y negativos.

El dataset para el ejercicio se describe [aquí](https://huggingface.co/datasets/somosnlp-hackathon-2023/suicide-comments-es).

In [ ]:
# Cargamos el DF con los datos positivos y negativos

df = pd.read_csv("hf://datasets/somosnlp-hackathon-2023/suicide-comments-es/suicide_comments_es.csv")
df

In [ ]:
# Extracción de datos positivos para análisis manual
df_positive = df[df['Label']==1]
df_positive.to_csv('positive.csv', index=False)
files.download('positive.csv')

# Extracción de datos negativos (por si es necesario analizarlos)
df_negative = df[df['Label']==0]


## Task 2: Extracción automática de características

Se aplicarán diferentes técnicas para extraer las características más relevantes de las notas de suicidio.

Comparar los resultados de estas técnicas con el conjunto de características que identificaron mediante su revisión manual de datos.

## Técnica 1: BOW

In [ ]:
# Preparación para extracción de características datos positivos

corpus = df_positive.Text.to_list()
#corpus

In [ ]:
# Instanciar el módulo para vectorizar los textos
vectorizador =  CountVectorizer()


# Vectorizar las palabras a partir de sus frecuencias
bow = vectorizador.fit_transform(corpus)

In [ ]:
# Vocabulario

print(vectorizador.vocabulary_)

In [ ]:
# Matriz de frecuencias

freqs = bow.sum(axis=0)
words = vectorizador.get_feature_names_out()

df_freqs = pd.DataFrame((freqs.T), index=words, columns=['freq'])
df_freqs.sort_values(by=['freq'], ascending=False)


In [ ]:
df_freqs.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8), title='Palabras más frecuentes en las notas de suicidio')
plt.legend(['Frecuencias'])
plt.savefig('absolute_freqs.png')
#files.download('absolute_freqs.png')

### Limpieza del dataset

En toda tarea que involucra el lenguaje natural es muy importante analizar qué palabras pueden aportar contenido, más allá de que puedan pertenecer a un listado de *stopwords*. Por ejemplo, en este caso, habría que ponderar si la palabra **no** debe ser parte de las *stopwords* o no. Dada la tarea, piensen en:

* no quiero vivir vs. [no] quiero vivir

In [ ]:
# Creamos una lista ad hoc de stopwords y la activamos en el vectorizador
# Si tenemos un archivo, cambiamos la variable por el nombre del archivo con las stopwords

stoplist = ['el', 'ella', 'a', 'ante', 'con', 'contra', 'que', 'de', 'un', 'una', 'pero', 'se', 'lo', 'en', 'ante', 'mi', 'la']

vectorizador_clean =  CountVectorizer(stop_words=stoplist, analyzer = 'word')
bow_clean = vectorizador_clean.fit_transform(corpus)
print(vectorizador_clean.vocabulary_)

In [ ]:
# Matriz de frecuencias

freqs_clean = bow_clean.sum(axis=0)
words_clean = vectorizador_clean.get_feature_names_out()
df_freqs_clean = pd.DataFrame((freqs_clean.T), index=words_clean, columns=['freq'])
df_freqs_clean.sort_values(by=['freq'], ascending=False)

In [ ]:
# Visualización de N palabras más frecuentes con corpus limpio

df_freqs_clean.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8), title='Palabras más frecuentes en las notas de suicidio sin stopwords')

plt.legend(['Frecuencias'])
plt.savefig('absolute_freqs_clean.png')
#files.download('absolute_freqs_clean.png')

## Técnica 2: TFIDF

In [ ]:
# Instanciamos el método

idf_vectorizer = TfidfVectorizer()

# Vectorizamos sobre el mismo conjunto de datos
corpus_idf = idf_vectorizer.fit_transform(corpus)


In [ ]:
# Vocabulario

print(idf_vectorizer.vocabulary_)


In [ ]:
# Matriz de frecuencias

freqs_idf = corpus_idf.sum(axis=0)
words_idf = idf_vectorizer.get_feature_names_out()

df_freqs_idf = pd.DataFrame((freqs_idf.T), index=words_idf, columns=['freq'])
df_freqs_idf.sort_values(by=['freq'], ascending=False)

In [ ]:
df_freqs_idf.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8), title='Palabras con TFIDF más frecuentes en las notas de suicidio', color='brown')

plt.legend(['Frecuencia ponderada por TFIDF'])
plt.savefig('freqs_idf.png')
#files.download('freqs_idf.png')

### ¿Limpieza del dataset?

Dado que la técnica de TF-IDF castiga las palabras que más se repiten en todo el corpus, por ejemplo, las stopwords o aquellas que son representativas de la temática del conjunto, conviene pensar si vale la pena aplicar quitar las stopwords.

Adicionalmente, hay que pensar en qué estamos representando: en el código anterior estamos trayendo las palabras más frecuentes (TF) después de aplicar el (IDF); no obstante, lo que vemos no son las palabras que mayor TFIDF tienen. Eso implica que hay que hacer una modificación en el código para traer aquellas que, sin ser necesariamente las de mayor frecuencia, sí tienen el mayor índice de (IDF). Por último, vamos a acativar la stoplist para ver qué tanto cambia la representación.



In [ ]:
# Primero instanciamos el método con las stopwords
idf_clean = TfidfVectorizer(stop_words=stoplist, analyzer = 'word')

# Vectorizamos
corpus_idf_clean = idf_clean.fit_transform(corpus)

# Después creamos la matriz ordenada por TFIDF
# En este caso, no traemos las palabras con mayor frecuencia, sino las que tienen mayor tfidf

freqs_idf_clean = corpus_idf_clean.max(axis=0).toarray().ravel()          #cambiamos el método .sum por .max para obtener las de mayor tfidf
words_idf_clean = idf_clean.get_feature_names_out()

df_freqs_idf_clean = pd.DataFrame((freqs_idf_clean), index=words_idf_clean, columns=['max_tfidf'])   #acá filtramos por mayor tfidf
df_freqs_idf_clean.sort_values(by=['max_tfidf'], ascending=False)

In [ ]:
# Visualización

df_freqs_idf_clean.sort_values(by=['max_tfidf'], ascending=False).head(30).plot(kind='bar', figsize=(15,8), title='Palabras con mayor TFIDF en las notas de suicidio', color='brown')

plt.legend(['Valor de TFIDF'])
plt.savefig('max_tfidf_clean.png')
#files.download('max_tfidf_clean.png')

## N-gramas

Con esta técnica vamos a utilizar valores de 2 a 4. Adicionalmente, vamos a restringir el número de características a los 1000 ngramas más representativos. Adicionalmente, estas características se extraerán considerando solo TF.

In [ ]:
# Se añade un parámetro para definir el rango de N

vectorizador_clean_n =  CountVectorizer(stop_words=stoplist, analyzer = 'word',
                                      ngram_range=(2,4), max_features=1000)
bow_clean_n = vectorizador_clean_n.fit_transform(corpus)


In [ ]:
# Guardamos los resultados en un nuevo DF

freqs_clean_n = bow_clean_n.sum(axis=0)
words_clean_n = vectorizador_clean_n.get_feature_names_out()
df_freqs_clean_n = pd.DataFrame((freqs_clean_n.T), index=words_clean_n, columns=['freq'])
df_freqs_clean_n.sort_values(by=['freq'], ascending=False)

In [ ]:
# Visualizamos

df_freqs_clean_n.sort_values(by=['freq'], ascending=False).head(30).plot(kind='bar', figsize=(15,8), title='N-gramas longitud 2:4 más frecuentes en las notas de suicidio', color='olive')

plt.legend(['Frecuencia de n-gramas'])
plt.savefig('ngrams_freqs.png')
#files.download('ngrams_freqs.png')

## N-gramas con TFIDF

Mantenemos los mismos valores y restricciones que con la técnica anterior.

In [ ]:
# Instanciamos el método
idf_clean_n =  TfidfVectorizer(stop_words=stoplist, analyzer = 'word',
                                      ngram_range=(2,4), max_features=1000)

# Vectorizamos
corpus_idf_clean_n = idf_clean_n.fit_transform(corpus)

# Creamos la matriz ordenada por TFIDF

freqs_idf_clean_n = corpus_idf_clean_n.max(axis=0).toarray().ravel()
words_idf_clean_n = idf_clean_n.get_feature_names_out()

df_freqs_idf_clean_n = pd.DataFrame((freqs_idf_clean_n), index=words_idf_clean_n, columns=['max_tfidf'])
df_freqs_idf_clean_n.sort_values(by=['max_tfidf'], ascending=False)

In [ ]:
# Visualizamos

df_freqs_idf_clean_n.sort_values(by=['max_tfidf'], ascending=False).head(30).plot(kind='bar', figsize=(15,8),
                                                                                  title='N-gramas longitud 2:4 con mayor valor de TFIDF en las notas de suicidio',
                                                                                  color='violet')
plt.legend(['TFIDF de n-gramas'])
plt.savefig('ngrams_idf.png')
#files.download('ngrams_idf.png')

## Comparación de características



In [ ]:
# Creamos una lista con los resultados de cada técnica
dfs = [df_freqs, df_freqs_clean, df_freqs_idf, df_freqs_idf_clean, df_freqs_clean_n, df_freqs_idf_clean_n]

# Creamos una lista vacía en donde se guardarán las características
features = []

# Iteramos sobre los DFs para traer las características
for i, df_temp in enumerate(dfs):
    col_name = df_temp.columns[0]
    df_sorted = df_temp.sort_values(by=col_name, ascending=False)
    features.append(pd.Series(df_sorted.index, name=f'Características{i+1}'))

# Creamos un DF para concatenar las características
df_comparisson = pd.concat(features, axis=1)
df_comparisson.columns = [
    'BOW',
    'BOW_clean',
    'TFIDF_x_Freq',
    'Max_TFIDF',
    'Ngrams',
    'Max_TFIDF_Ngrmas'
]

display(df_comparisson.head(30))

df_comparisson.to_csv('features.csv', index=False)
#files.download('features.csv')

**Nota**: recordar que en la técnica de N-grams se hizo la reducción a las 1000 características más importantes, criterio que no se aplicó a las primeras técnicas. Si bien esto no afecta el resumen de características de este DF, sí es conveniente que se establezcan los mismos criterios para todas las técnicas, de forma que, si guardamos el DF en un archivo externo, se registre la misma información siempre, i. e. las mismas n-características para todas las técnicas.

## Task 2: clasificación textual

Esta tarea de PLN busca que un modelo X identifique las características más discriminantes entre clases (en el caso del ejercicio, dos clases) para clasificar un texto como nota de suicidio (dato positivo) o de no suicidio (dato negativo).

El escenario que supone la tarea implica que hay que trabajar con datos positivos y negativos al mismo tiempo. Ello signfica que los documentos que tienen la etiqueta 1 deben ser lo suficientemente representativos de lo que puede entenderse como una nota de suicidio. Asimismo, los datos negativos deben ser claros ejemplos de aquello que no es una nota de suicidio. En conjunto, tanto datos positivos como negativos deben tender a discriminarse entre sí.

Ahora bien, en el contexto de clasificación textual, el modelo debe aprender claramente qué caracteriza a los datos positivos para poder separarlos de los negativos. Para ello, en PLN se acostumbra dividir el dataset en dos particiones: **training** y **test**.

El training sirve para observar cómo se comporta el modelo en la extracción de características, en tanto que el test sirve para evaluar la calidad del modelo. Las métricas que hemos visto en clase se aplican al desempeño que tiene el modelo en la partición de test.

Nota: se acostumbra dividir el dataset en 70% training y 30% test. Finalmente, señalar que el test nunca se ve, solo sirve para evaluar datos que el modelo no conoce.

In [ ]:
# Revisamos la distribución de clases en el dataset

print(df.Label.value_counts())

### Primera implicación

Notar que la distribución de datos positivos y negativos no está balanceada. Hay una relación de 9:1; es decir, por cada 10 notas, hay alrededor de nueve notas que no son de suicidio por una que sí lo es. Esto supone un escenario típico de PLN, pues no se espera que haya una distribución balanceada de este tipo de notas en contextos reales.

En este tipo de escenarios, cobra mayor relevancia la extracción de características, puesto que estas deben ser altamente discriminantes para poder reconocer aquello que, de facto, es un dato poco común.

In [ ]:
# Dividimos el DF en training y test usando el método train_test_split de sklearn

x_train, x_test, y_train, y_test = train_test_split(df.Text, df.Label, test_size=0.3, random_state=42)

# Ver el tamaño de documentos de cada partición
print(f'Training: {x_train.shape}, Test: {x_test.shape}')


In [ ]:
# Verificamos cómo queda la partición de test en términos de muestras positivas y negativas

print('Tamaño de clases en training:')
print(y_train.value_counts())


## Entrenamiento de modelo de clasificación

Vamos a utilizar el conjunto de características que recién obtuvimos para modelar qué es una nota de suicidio. La hipótesis es que, si las características identificadas mediante los métods antes expuestos son los suficientemente discriminantes del fenómeno, entonces el modelo de clasificación podrá tener un comportamiento correcto para distinguir una nota de suicidio de una que no lo es.

El modelo será evaluado con base en las métricas trabajadas en clase: accuracy, precision, recall y F1.



In [ ]:
# Importamos el módulo de clasificación de Bayes
from sklearn.naive_bayes import MultinomialNB


Notar que hay diferentes tipos de clasificadores: Bayes, árboles de decisión, máquinas de soporte vectorial, entre otros. Cada clasificador tiene, a su vez, implementaciones más específicas que permiten el acercamiento y modelado de la tarea considerando diferentes criterios. En este sentido, la librería de sklearn tiene implementados diferentes algoritmos para realizar tareas de clasificación. En esta [página](https://scikit-learn.org/stable/supervised_learning.html) pueden encontrar la descripción de varios de ellos y la forma en la que se implementan.

Adicionalmente, hay que tomar en cuenta cuál es la tarea que se busca modelar, así como el tipo de dato para elegir el mejor clasificador. Para el caso de esta tarea, optamos por un clasificador bayesiano (el cual es bueno para trabajar con frecuencias de características). Este clasificador tiene como base el **Teorema de Bayes**, el cual permite calcular la probabilidad de que algo sea cierto, sabiendo que otra cosa ya ha sucedido; es decir, calcula que tan probable es que suceda B, sabiendo que ya sucedió A. Por ejemplo, qué tan probable es que *favor* aparezca si antes apareció *por*.

La fórmula del teorema de Bayes es la siguiente:

- $P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$






In [ ]:
# Definimos las técnicas que usaremos para la extracción de características

vectorizadores = {
    "BOW": CountVectorizer(stop_words=stoplist, analyzer = 'word', max_features=1000),
    "TFIDF": TfidfVectorizer(stop_words=stoplist, analyzer = 'word', max_features=1000),
    "N-grams": CountVectorizer(stop_words=stoplist, analyzer = 'word', ngram_range=(2,4), max_features=1000),
    "IDF-grams": TfidfVectorizer(stop_words=stoplist, analyzer = 'word', ngram_range=(2,4), max_features=1000)
}

In [ ]:
# Clasificador con BOW

# Vectorización de training con la técnica de BOW llamando el vectorizador por su ID
x_train_vec = vectorizadores["BOW"].fit_transform(x_train)

# Vectorización de test
# Nota: no usar NUNCA el método fit en test, ese es solo para training
x_test_vec = vectorizadores["BOW"].transform(x_test)

# Inicializar el clasificador
# Nota: tal como se mencionó, usaremos el algoritmo de Bayes. La forma de llamarlo en sklearn es MultinomialNB()
# Nota: para el modelado, usamos ambas particiones (x,y: training,test)

modelo_bow = MultinomialNB()
modelo_bow.fit(x_train_vec, y_train)

# Clasificación
y_pred = modelo_bow.predict(x_test_vec)

# Resultados
acc = accuracy_score(y_test, y_pred)
print('\n' + '='*50)
print(f' Técnica BOW')
print('='*50)

print(f'Accuracy: {acc:.4f}\n')
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred))


# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['No suicidio (0)', 'Suicidio (1)'],
                yticklabels=['No Suicidio (0)', 'Suicidio (1)'])
plt.title(f'Matriz de confusión BOW')
plt.ylabel('Clase real ')
plt.xlabel('Clase del modelo')
plt.show()


In [ ]:
# Clasificador con TFIDF

# Vectorización de training con la técnica de TFIDF
x_train_vec = vectorizadores["TFIDF"].fit_transform(x_train)

# Vectorización de test
# Nota: no usar NUNCA el método fit en test, ese es solo para training
x_test_vec = vectorizadores["TFIDF"].transform(x_test)

# Inicializar el clasificador
modelo_bow = MultinomialNB()
modelo_bow.fit(x_train_vec, y_train)

# Clasificación
y_pred = modelo_bow.predict(x_test_vec)

# Resultados
acc = accuracy_score(y_test, y_pred)
print('\n' + '='*50)
print(f' Técnica TFIDF')
print('='*50)

print(f'Accuracy: {acc:.4f}\n')
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred))


# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False,
                xticklabels=['No suicidio (0)', 'Suicidio (1)'],
                yticklabels=['No Suicidio (0)', 'Suicidio (1)'])
plt.title(f'Matriz de confusión TFIDF')
plt.ylabel('Clase real ')
plt.xlabel('Clase del modelo')
plt.show()

In [ ]:
# Clasificador con N-gramas (2-4)

# Vectorización de training con la técnica de N-gramas
x_train_vec = vectorizadores["N-grams"].fit_transform(x_train)

# Vectorización de test
# Nota: no usar NUNCA el método fit en test, ese es solo para training
x_test_vec = vectorizadores["N-grams"].transform(x_test)

# Inicializar el clasificador
modelo_bow = MultinomialNB()
modelo_bow.fit(x_train_vec, y_train)

# Clasificación
y_pred = modelo_bow.predict(x_test_vec)

# Resultados
acc = accuracy_score(y_test, y_pred)
print('\n' + '='*50)
print(f' Técnica N-gramas')
print('='*50)

print(f'Accuracy: {acc:.4f}\n')
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred))


# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', cbar=False,
                xticklabels=['No suicidio (0)', 'Suicidio (1)'],
                yticklabels=['No Suicidio (0)', 'Suicidio (1)'])
plt.title(f'Matriz de confusión N-gramas')
plt.ylabel('Clase real ')
plt.xlabel('Clase del modelo')
plt.show()

In [ ]:
# Clasificador con TFIDF de N-gramas

# Vectorización de training con la técnica de TFIDF de n-gramas
x_train_vec = vectorizadores["IDF-grams"].fit_transform(x_train)

# Vectorización de test
# Nota: no usar NUNCA el método fit en test, ese es solo para training
x_test_vec = vectorizadores["IDF-grams"].transform(x_test)

# Inicializar el clasificador
modelo_bow = MultinomialNB()
modelo_bow.fit(x_train_vec, y_train)

# Clasificación
y_pred = modelo_bow.predict(x_test_vec)

# Resultados
acc = accuracy_score(y_test, y_pred)
print('\n' + '='*50)
print(f' Técnica TFIDF de N-gramas')
print('='*50)

print(f'Accuracy: {acc:.4f}\n')
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred))


# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Grays', cbar=False,
                xticklabels=['No suicidio (0)', 'Suicidio (1)'],
                yticklabels=['No Suicidio (0)', 'Suicidio (1)'])
plt.title(f'Matriz de confusión TFIDF de N-gramas')
plt.ylabel('Clase real ')
plt.xlabel('Clase del modelo')
plt.show()



---








---



Si queremos correr todos los experimentos en un solo bloque, añadimos una secuencia de control para que itere sobre la información de los vectorizadores y, dentro de esa misma secuencia, incorporamos los procesos que realizamos de forma independiente.

Nota: revisar la asignación de variables para que no haya problemas, así como el indentado para la secuencia de control.

In [ ]:
# Proceso global en una sola fase con iteración

# Definimos un diccionario de colores para cada técnica
colores_map = {
    "BOW": "Blues",
    "TFIDF": "Greens",
    "N-grams": "Reds",
    "IDF-grams": "Grays"
}

# Definimos un bucle con el nombre de cada técnica y su vectorizador
for name, vec in vectorizadores.items():
    print("\n" + "="*50)
    print(f" Técnica: {name}")
    print("="*50)

    # Se vectorizan los datos de training
    x_train_vec = vec.fit_transform(x_train)

    # SE Vectoriza los datos de test
    x_test_vec = vec.transform(x_test)

    # Inicializar el clasificador
    modelo = MultinomialNB()
    modelo.fit(x_train_vec, y_train)

    # Clasificamos
    y_pred = modelo.predict(x_test_vec)

    # Resultados por técnica
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}\n")
    print("Reporte de clasificación:")
    print(classification_report(y_test, y_pred))

    # Obtener el mapa de colores asignado o usar uno por defecto
    color_elegido = colores_map.get(name, "Purples")

    # Graficar matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=color_elegido, cbar=False,
                xticklabels=['No Suicida (0)', 'Suicida (1)'],
                yticklabels=['No Suicida (0)', 'Suicida (1)'])
    plt.title(f'Matriz de confusión - {name}')
    plt.ylabel('Clase real')
    plt.xlabel('Clase asignada por el modelo')

    plt.savefig(f'confusion_matrix_{name}.png')
    #files.download(f'confusion_matrix_{name}.png')

    plt.show()